# Burel — launcher Colab

LLM privato con architettura HOPE / nested learning a pieno giro.

**Prima di iniziare:** `Runtime → Cambia tipo di runtime → GPU` (T4 va bene).

Carica `Burel.zip` nella tua cartella Drive, poi esegui le celle in ordine. I checkpoint
(`burel_best.pt` / `burel_last.pt`) vengono salvati accanto allo zip: se Colab si
disconnette, rilancia la cella di training e riprende da solo.

## 1. Monta Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Trova `Burel.zip` nel Drive e scompatta
Cerca lo zip in tutto `MyDrive`. Se l'hai messo in una sottocartella specifica e vuoi
accelerare, imposta `SEARCH_ROOT` su quel percorso.

In [ ]:
import glob, os, shutil, zipfile

SEARCH_ROOT = '/content/drive/MyDrive'  # cartella in cui hai caricato Burel.zip

hits = glob.glob(os.path.join(SEARCH_ROOT, '**', 'Burel.zip'), recursive=True)
assert hits, 'Burel.zip non trovato in Drive. Caricalo e riprova.'
zip_path = hits[0]
DRIVE_DIR = os.path.dirname(zip_path)
print('Trovato:', zip_path)
print('Cartella Drive:', DRIVE_DIR)

if os.path.exists('/content/Burel'):
    shutil.rmtree('/content/Burel')
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content')
assert os.path.exists('/content/Burel/scripts/train.py'), 'Struttura zip inattesa: manca /content/Burel/scripts/train.py'
print('Scompattato in /content/Burel')

## 3. Installa il package + punta i checkpoint sul Drive
`pip install -e .` installa Burel e le sue dipendenze. I `.pt` finiscono in
`<cartella>/Burel_checkpoints`, così sopravvivono alle disconnessioni.

In [ ]:
%cd /content/Burel
!pip -q install -e .

import yaml
cfg_path = 'configs/config.yaml'
cfg = yaml.safe_load(open(cfg_path))
cfg['train']['drive_backup_dir'] = os.path.join(DRIVE_DIR, 'Burel_checkpoints')
cfg.setdefault('data', {})['drive_cache_dir'] = os.path.join(DRIVE_DIR, 'Burel_data')
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False, allow_unicode=True)
print('Checkpoint su Drive:', cfg['train']['drive_backup_dir'])
print('Dataset cache su Drive:', cfg['data']['drive_cache_dir'])

## 4. Prepara il dataset
Il dataset attivo lo decide `configs/config.yaml` (`data.name`): di default
`tinystories` (BPE byte-level, fase 2). La prima volta scarica TinyStories da
HuggingFace, allena il tokenizer BPE e scrive `train.bin`/`val.bin` + `tokenizer.json`
in `data_cache/`. E' idempotente: se la config dati non cambia, salta il lavoro.

In [ ]:
!python scripts/prepare_data.py

## 5. Training
Barra di avanzamento con %, ETA e loss. Riprende in automatico da `burel_last.pt`
(locale o Drive) se esiste. Per misurare la velocità prima di un run lungo, abbassa
`max_iters` in `configs/config.yaml` (es. 300).

In [ ]:
!python scripts/train.py

## 6. Inferenza — genera testo

In [ ]:
!python scripts/generate.py --prompt "ROMEO:" --tokens 600 --temperature 0.7 --top_k 100

## (facoltativo) Verifica causalità stretta

In [ ]:
!python tests/test_causal.py